In [4]:

import numpy as np
import global_config as cfg
from gpt_library import GPTLibrary
import pickle
import train_funcs as tf
import os


def run_train_pipeline(lib, paradigm, mup=False, train_bestest=False):
    '''wrapper to coordinate non notebook training fns in train_funcs.py'''

    # load bin sets
    train = np.memmap(cfg.TRAIN_TOKEN_PATH, dtype=np.uint16, mode='r')
    val = np.memmap(cfg.VAL_TOKEN_PATH, dtype=np.uint16, mode='r')

    # determine which config set to use and handle LR Sweeps
    if mup:
        # Use the width-scaling reference (tiny) for the LR sweep
        sweep_results, best_lr = tf.lr_sweep(cfg.MUP_CONFIG["m_tiny"], train)
        mod_configs = cfg.MUP_CONFIG
        file_pref = "mup"
    elif train_bestest:
        mod_configs = cfg.BESTEST_CONFIG
        file_pref = "bestest"
        best_lr = cfg.BEST_LR
    else:
        sweep_results, best_lr = tf.lr_sweep(cfg.STANDARD_CONFIG["tiny"], train)
        mod_configs = cfg.STANDARD_CONFIG
        file_pref = "standard"
    #safe print sweep
    if not train_bestest:
        print("-" * 30)
        print(f"Sweep Results (LR, Avg Loss): {sweep_results}")
        print(f"Optimal Learning Rate identified: {best_lr}")
        print("-" * 30)

    # main Training Loop
    for key, config_obj in mod_configs.items():
        tf.run_model_stage(
            key, train, val, config_obj, lib,
            best_lr, file_pref, train_bestest=train_bestest
        )
        print(f"Completing training on model {key}")

    # we will persist
    save_path = os.path.join(cfg.SAVE_DIR, f'{paradigm}_training_library.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(lib, f)

    print(f"Library object frozen and saved to {save_path}")



#example usage
#gpt_library = GPTLibrary(cfg.BESTEST_CONFIG)
#run_train_pipeline(gpt_library, "gpt", mup=False, train_bestest=True)



--- Using AdamW | LR: 0.001585 ---
scheduler done gotted

 Training BESTEST | Params: 39,979,008 | Steps: 7685 | LR: 0.001585


KeyboardInterrupt: 